### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -U langchain-text-splitters langchain-community pymupdf

In [ ]:
from unsloth import FastLanguageModel
import torch
import requests
from langchain_community.document_loaders import PyMuPDFLoader
import pandas as pd
from tqdm import tqdm
import torch
from transformers import TextStreamer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.4.8 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [14]:
url = 'https://www.lobbyregister.bundestag.de/media/2c/d5/710485/SAP-Integrated-Report-2025.pdf'

loader = PyMuPDFLoader(url)
docs = loader.load()

data = pd.DataFrame()
for i in range(len(docs)):
    data.at[i, 'page'] = int(docs[i].metadata['page'])
    data.at[i, 'text'] = docs[i].page_content

In [ ]:
# for txt document
# text_splitter = CharacterTextSplitter(
#    separator="\n\n",
#    chunk_size=1000,
#    chunk_overlap=50,
#    length_function=len,
#    is_separator_regex=False
# )

# texts = text_splitter.split_text(content)

In [ ]:
system_prompt = {
    'name': 'system_prompt_v2', 'text':'''
    Respond terse like smart caveman.
    Drop all fluff.
    Use short words.
    Technical substance stay.
    Minimum tokens.
'''
}

In [16]:
MAX_NEW_TOKENS = 2056
TEMPERATURE = 1
TOP_P = 0.95
TOP_K = 20

In [ ]:
data['temp_1.0'] = None
for i, ro in tqdm(data.iterrows(), total=len(data)):
    text = ro['text']

    # handling empty text
    if pd.isna(text) or not text.strip():
      continue

    messages = [{"role": "system", "content": system_prompt['text']},
                {"role": "user", "content": f"{text}"}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # Must add for generation
        enable_thinking=False,        # Enable thinking
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    generated_ids = model.generate(
    **inputs,
    max_new_tokens = MAX_NEW_TOKENS,           # Increase for longer outputs!
    temperature = TEMPERATURE,
    top_p = TOP_P,
    top_k = TOP_K,
    do_sample = True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

    generated = tokenizer.decode(
    generated_ids[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
    )

    data.at[i, 'temp_1.0'] = generated
data.to_csv(f'SAP_facts.csv')

  0%|          | 0/327 [00:00<?, ?it/s]

1. The report is for the year 2025.  
2. The report is an Integrated Report by SAP.<|im_end|>


  0%|          | 1/327 [00:06<36:51,  6.78s/it]

- The SAP Integrated Report 2025 combines financial, environmental, social, and governance performance.  
- The report is available at www.sapintegratedreport.com.  
- The TCFD recommendations are embedded on the Integrated Report homepage.  
- The report is prepared in accordance with the German Commercial Code and German Accounting Standards.  
- The Combined Management Report complies with IFRS Practice Statement Management Commentary.  
- The report includes SAP SE and all subsidiaries controlled by SAP, as per IFRS.  
- Joint arrangements and associates are excluded from the sustainability reporting.  
- The Group Sustainability Statement follows the European Sustainability Reporting Standards (ESRS).  
- Greenhouse gas data is based on the Greenhouse Gas Protocol.  
- All data is sourced from SAP software solutions and responsible business units.  
- The reporting period is fiscal year 2025.  
- The report includes information up to February 18, 2026.  
- The report is available 

  1%|          | 2/327 [00:42<2:10:25, 24.08s/it]

...

- SAP SE is required to produce a non-financial statement (NFS) for the holding company under section 289b–e of the German Commercial Code (HGB).  
- SAP SE must also prepare a non-financial group statement under section 315b–c in conjunction with section 289c–e of the German Commercial Code (HGB).  
- The Group Sustainability Statement fulfills both the non-financial statement and non-financial group statement requirements.  
- The report is prepared in accordance with the European Sustainability Reporting Standards (ESRS).  
- SAP chose ESRS to prepare for the transposition of the EU Corporate Sustainability Reporting Directive (CSRD) into German law.  
- ESRS take precedence over reporting principles in German Accounting Standard (GAS) 20 until CSRD is transposed into German law.  
- The Group Sustainability Statement includes sections on General Information, Environmental Information, Social Information, and Governance Information.  
- General Information includes subsections on Su

 30%|███       | 99/327 [1:36:31<3:03:23, 48.26s/it]

1. SAP's non-financial disclosures are determined through a materiality assessment using internal and external inputs.  
2. No material risks according to section 289c (3) sentence numbers 3 and 4 HGB have been identified.  
3. Environmental matters include climate change and resource use, with a focus on greenhouse gas emissions.  
4. SAP addresses employee matters, including workforce policies, inclusion, health, and development.  
5. A key performance indicator for employees is the Employee Engagement Index.  
6. SAP reports on social matters, including workers in the value chain, data protection, and responsible AI.  
7. A key performance indicator for customer satisfaction is the Cloud Customer Satisfaction (Cloud CSAT).  
8. SAP includes human rights due diligence in its non-financial disclosures.  
9. Anti-corruption and bribery matters are addressed under business conduct policies.  
10. The report includes references to consolidated financial statements and related notes.  
11

 31%|███       | 100/327 [1:37:11<2:53:05, 45.75s/it]

- The Group Sustainability Statement is prepared on a consolidated basis, with the same scope as the financial statements.  
- Data points derived from other EU legislation and required by ESRS are presented in the Appendix.  
- Time horizons are defined as short term (up to one year), medium term (more than one and up to five years), and long term (more than five years).  
- Unless stated otherwise, SAP’s entire up- and downstream value chain is included in the Group Sustainability Statement.  
- Information classified as sensitive or related to intellectual property is omitted and noted in the respective sections.  
- All actions presented are ongoing unless otherwise stated.  
- Metrics are not audited or validated by external experts other than the assurance provider.  
- Acquired companies not fully integrated into SAP policies are excluded from certain metrics.  
- Metrics requiring a gender breakdown exclude employees with gender recorded as "Other", "Not Reported", or "Not Disc